# Fine-Tune LFM3-VL-2B for Wind Turbine Blade Inspection with TRL (SFT)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Liquid4All/cookbook/blob/main/examples/wind-turbine-inspection/notebooks/lfm3_vl_sft.ipynb)

This notebook teaches [LFM3-VL-2B](https://huggingface.co/LiquidAI/LFM3-VL-2B) to find surface defects on wind turbine blade photos and name their type. It uses supervised fine-tuning (SFT) with [TRL](https://github.com/huggingface/trl) and LoRA on the [WTBs2025](https://doi.org/10.6084/m9.figshare.28876391) dataset, and returns every defect as JSON with a class and a bounding box.

Out of the box, LFM3-VL-2B names the right defect type on about 9% of test photos, close to chance for nine types. Fine-tuning teaches it what each defect looks like. This is the first of two steps: [`lfm3_vl_grpo.ipynb`](lfm3_vl_grpo.ipynb) continues from the adapter trained here with reinforcement learning.

## 🎯 What you'll find

- **The task**: a blade photo goes in, a JSON list of `{"label", "bbox_2d"}` objects comes out
- **Balanced training data**: a per-class cap, so the model doesn't learn to answer the most common defect type
- **LoRA + SFT**: train the language model and the projector on the answer only, with the vision encoder frozen
- **Evaluation**: detection F1, box-only F1, type accuracy and recall by defect size on the validation split

**Training matches evaluation exactly:**
- **Same prompt:** the task text and answer format from `blade_eval.build_prompt("xyxy1000")`, which [`evaluate.ipynb`](evaluate.ipynb) sends to every model.
- **Same answer format:** `[{"label": ..., "bbox_2d": [x_min, y_min, x_max, y_max]}]` on a 0–1000 grid. The base model already answers in this format, so training builds on it instead of teaching a new syntax.
- **Same images:** full resolution, as in the evaluation.
- **No leakage:** training uses the `train` split only. Each original photo and its augmented copies sit in one split, and the `test` split stays held out.

**Hardware:** a CUDA GPU with bfloat16 support. On Colab, use an A100 (an L4 works with a smaller batch size). Two epochs take about 21 minutes on an A100.

## Install required packages

In [ ]:
# The versions this notebook was tested with. PEFT needs torchao >= 0.16; Colab ships an older one.
%pip install -q "transformers==5.20.0" "trl==1.16.0" "peft==0.21.2" "torchao==0.18.0" datasets accelerate pillow pandas matplotlib

## Hugging Face token

You need a token with write access to push the trained adapter to the Hub, where [`lfm3_vl_grpo.ipynb`](lfm3_vl_grpo.ipynb) picks it up. On Colab, add it as the `HF_TOKEN` secret (key icon in the sidebar) and grant this notebook access when asked.

In [ ]:
import os

from huggingface_hub import get_token

try:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass  # not on Colab, or no secret

print("HF token found" if get_token() else "No HF token: you won't be able to push the adapter to the Hub")

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU: switch the runtime to a GPU (Runtime > Change runtime type)."
print(f"🖥️  GPU: {torch.cuda.get_device_name(0)} | bf16: {torch.cuda.is_bf16_supported()}")

## Load the shared evaluation code

`blade_eval.py` holds the dataset loader, the deterministic split, the prompt and the scorer. The training notebooks and `evaluate.ipynb` all use it, so every result is computed the same way. On Colab, upload it from the example folder when asked.

In [ ]:
import os
import sys

# Locally, blade_eval.py is one folder up. On Colab, upload it next to the notebook.
if os.path.exists("../blade_eval.py"):
    sys.path.insert(0, "..")
elif not os.path.exists("blade_eval.py"):
    raise FileNotFoundError("Upload blade_eval.py from the example folder next to this notebook.")

import blade_eval as be


## Configuration

`MAX_PER_CLASS` balances the defect types. The training split is dominated by erosion (1,945 images, against 125 for oil leakage), so without a cap the model would learn to answer "erosion" too often. Capped classes are sampled at random; smaller classes keep all their images.
During training, the validation loss is computed on 200 fixed validation images every 100 optimizer steps and once at the end. Live logs include loss, gradient norm, learning rate and GPU memory. To pick a checkpoint, use detection F1 on all 720 validation images instead.

Set `HUB_REPO` to push the adapter to the Hub; the GRPO notebook loads it from there.


In [ ]:
import transformers

SEED = 42
transformers.set_seed(SEED)

MODEL_ID = "LiquidAI/LFM3-VL-2B"
BASE_MODEL_INFO = be.model_identity(MODEL_ID)  # the model's current commit, recorded with every result
OUTPUT_DIR = "./outputs/lfm3-vl-blade-sft"  # use a new directory for each run
BOX_FORMAT = "xyxy1000"  # the answer format the evaluation uses for LFM3-VL
MAX_PER_CLASS = 600  # training images per defect type; None uses the whole training split
MAX_IMAGE_TOKENS = None  # None keeps full resolution, as in the evaluation
N_VAL_EVAL = None  # score all 720 validation images after training

# Training
NUM_EPOCHS = 2
BATCH_SIZE = 4  # lower to 2 on an L4
GRAD_ACCUMULATION = 4  # effective batch size = BATCH_SIZE * GRAD_ACCUMULATION
LEARNING_RATE = 2e-4
LOGGING_STEPS = 10
EVAL_STEPS = 100  # optimizer steps; about four checks during this two-epoch run

# Set to a repo id (e.g. "your-username/lfm3-vl-blade-sft") to push the LoRA adapter; the GRPO notebook starts from it.
HUB_REPO = None

DATA_DIR = "./data"

## Load the Training Data

The first run downloads WTBs2025 (~480 MB). The `train` split keeps the dataset's augmented copies (flips, rotations, brightness changes and GAN images for lightning strikes; repeated exports for oil leakage), which add variety for training. Validation and test use original images only.

In [ ]:
import collections
import random

root = be.download(DATA_DIR)

# Identical photos occur in both the lightning-strike and pinhole folders with partial labels.
# Drop both sides and all augmented copies of a matching original from training/validation.
train_items = be.load_items(root, split="train", exclude_cross_class_duplicates=True)
val_items = be.load_items(root, split="val", exclude_cross_class_duplicates=True)
assert len(train_items) == 4915, f"Expected 4,915 training images before balancing, got {len(train_items)}. Check blade_eval.py and dataset version."
assert len(val_items) == 720, f"Expected 720 validation images, got {len(val_items)}. Check blade_eval.py and dataset version."
print(f"Training split: {len(train_items)} images")
print(f"Validation split: {len(val_items)} images")
print(f"Excluded cross-class groups: {len(be.cross_class_duplicate_groups(root.resolve()))}")

rng = random.Random(SEED)
by_class = collections.defaultdict(list)
for item in train_items:
    by_class[item.label].append(item)
train_items = [
    item
    for items in by_class.values()
    for item in (rng.sample(items, MAX_PER_CLASS) if MAX_PER_CLASS and len(items) > MAX_PER_CLASS else items)
]
rng.shuffle(train_items)
assert len(train_items) == 3315, f"Expected 3,315 balanced training images, got {len(train_items)}."
print(f"After balancing: {len(train_items)} images")
print(dict(collections.Counter(item.label for item in train_items)))


## Build the Training Examples

Each example is the evaluation prompt plus the ground-truth boxes as the answer (`be.format_boxes`), ordered top to bottom. Images are loaded in the collate function, so the dataset only holds file paths.

In [ ]:
from datasets import Dataset

PROMPT = be.build_prompt(BOX_FORMAT)


def to_rows(items):
    return Dataset.from_list(
        [{"item_id": item.item_id, "path": str(item.path), "answer": be.format_boxes(item.boxes)} for item in items]
    )


train_ds = to_rows(train_items)
eval_ds = to_rows(val_items[:200])  # validation loss during training
print(PROMPT)
print("\nAnswer:", train_ds[0]["answer"])

Some training examples, with the target boxes in green:

In [ ]:
import matplotlib.pyplot as plt

shown = list({item.label: item for item in train_items}.values())
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for ax in axes.flat:
    ax.axis("off")
for ax, item in zip(axes.flat, shown):
    ax.imshow(be.draw(item))
    ax.set_title(item.label)
plt.tight_layout()
plt.show()

## Load the Model

bf16 on a CUDA GPU. Images stay at full resolution unless `MAX_IMAGE_TOKENS` is set: each LFM3-VL image token covers a 32×32 pixel block, so a 640×640 image is about 400 tokens and a 1280×1080 oil leakage image about 1,350.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL_ID, revision=BASE_MODEL_INFO["revision"], trust_remote_code=True)
if MAX_IMAGE_TOKENS:
    processor.image_processor.size["longest_edge"] = MAX_IMAGE_TOKENS * 32 * 32

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, revision=BASE_MODEL_INFO["revision"], dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)
print(f"🔢 Parameters: {model.num_parameters():,}")

## Sanity check before training

Long inputs, like the 1280×1080 oil leakage photos, can break bfloat16 generation on some hardware: the model answers `[{"` and then only newlines. This checks one oil leakage photo and one 640 px photo. If the oil leakage answer is broken, set `MAX_IMAGE_TOKENS = 1024` and reload the model, so training doesn't learn from inputs the model can't process.

In [ ]:
def answer(model, item, max_new_tokens=256):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT}]}]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = processor(text=[prompt], images=item.image(), return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.decode(output[0, inputs["input_ids"].shape[1] :], skip_special_tokens=True)


check_items = [
    next(item for item in val_items if item.label == "oil leakage"),
    next(item for item in val_items if item.label == "lightning strikes"),
]
for item in check_items:
    print(f"--- {item.item_id} ({item.width}x{item.height})")
    print("Target:", be.format_boxes(item.boxes))
    print("Model: ", answer(model, item)[:300])

## Format Conversations

Each example is a user turn (image and prompt) and an assistant turn (the boxes). The assistant turn sets `reasoning_content` to an empty string, so the chat template renders it as `<think></think>answer`, matching inference with `enable_thinking=False`.

The collate function uses `return_assistant_tokens_mask=True`, so only the answer tokens are trained on; the prompt and image tokens are masked with `-100`.

In [ ]:
from PIL import Image


def to_messages(sample: dict) -> list[dict]:
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": Image.open(sample["path"]).convert("RGB")},
                {"type": "text", "text": PROMPT},
            ],
        },
        {"role": "assistant", "reasoning_content": "", "content": [{"type": "text", "text": sample["answer"]}]},
    ]


def collate_fn(samples):
    batch = processor.apply_chat_template(
        [to_messages(sample) for sample in samples],
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        return_assistant_tokens_mask=True,
        processor_kwargs={"padding": True},
    )
    assistant_mask = batch.pop("assistant_masks").bool()
    batch["labels"] = batch["input_ids"].masked_fill(~assistant_mask, -100)
    return batch


example = collate_fn([train_ds[0]])
print(f"Sequence length: {example['input_ids'].shape[1]} tokens")
print("Trained on:", repr(processor.decode(example["input_ids"][0][example["labels"][0] != -100])))

## Configure LoRA

LoRA targets the language model's gated short convolutions, its attention projections (including the per-head output gate `gate_proj`) and its MLP, plus the multimodal projector that maps image features into the language model. The regex is anchored to `language_model` and `multi_modal_projector`, so the vision encoder, which reuses some of the same module names, stays frozen.

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

LM_MODULES = "w1|w2|w3|q_proj|k_proj|v_proj|o_proj|gate_proj|in_proj|out_proj"
PROJECTOR_MODULES = "linear_1|linear_2"

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=rf".*language_model.*\.({LM_MODULES})|.*multi_modal_projector\.({PROJECTOR_MODULES})",
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Train

Save a checkpoint after each epoch. Pick the adapter by detection F1 on the validation split, not by training loss: score the saved `checkpoint-*` adapters on the same 720 validation images. The cell stops if `OUTPUT_DIR` already exists, so a rerun never overwrites an earlier adapter.

The live JSONL log records each training and validation event. An evaluation also runs at step 0 and at the end, so the chart shows a before/after comparison even when the run ends between scheduled checks. Validation loss measures teacher-forced target text; it is a useful training-health signal, while box F1 measures the actual task.


In [ ]:
from pathlib import Path
assert not Path(OUTPUT_DIR).exists(), f"{OUTPUT_DIR} already exists; choose a new OUTPUT_DIR or deliberately resume instead of overwriting."

from trl import SFTConfig, SFTTrainer
from transformers import TrainerCallback
from datetime import datetime, timezone
import json
import math
import time


class TrainingHealthCallback(TrainerCallback):
    def __init__(self, output_dir):
        self.path = Path(output_dir) / "training_log.jsonl"
        self.started = time.monotonic()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not state.is_world_process_zero or not logs:
            return control
        row = {
            "time_utc": datetime.now(timezone.utc).isoformat(),
            "step": state.global_step,
            "epoch": state.epoch,
            "elapsed_minutes": round((time.monotonic() - self.started) / 60, 2),
            **logs,
        }
        if torch.cuda.is_available():
            row["gpu_allocated_gb"] = round(torch.cuda.memory_allocated() / 1024**3, 2)
            row["gpu_reserved_gb"] = round(torch.cuda.memory_reserved() / 1024**3, 2)
            row["gpu_peak_allocated_gb"] = round(torch.cuda.max_memory_allocated() / 1024**3, 2)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        with self.path.open("a") as stream:
            safe_row = {key: (str(value) if isinstance(value, float) and not math.isfinite(value) else value) for key, value in row.items()}
            stream.write(json.dumps(safe_row, default=str, allow_nan=False) + "\n")
        fields = ("loss", "eval_loss", "grad_norm", "learning_rate", "gpu_allocated_gb")
        detail = "  ".join(f"{key}={row[key]:.4g}" for key in fields if isinstance(row.get(key), (int, float)))
        print(f"[health] step={state.global_step} epoch={state.epoch or 0:.2f}  {detail}")
        for key in ("loss", "eval_loss", "grad_norm"):
            value = row.get(key)
            if isinstance(value, (int, float)) and not math.isfinite(value):
                print(f"WARNING: {key} is non-finite at step {state.global_step}; inspect this run before using its adapter.")
                control.should_training_stop = True
        return control

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="linear",
    warmup_steps=0.1,  # transformers 5 dropped `warmup_ratio`: a float < 1 is a ratio of total steps
    weight_decay=0.01,
    gradient_checkpointing=True,
    bf16=True,
    logging_steps=LOGGING_STEPS,
    logging_first_step=True,
    logging_nan_inf_filter=False,  # expose unstable loss instead of hiding it
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    eval_on_start=True,
    prediction_loss_only=True,
    save_strategy="epoch",
    save_total_limit=NUM_EPOCHS,
    report_to="none",
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},  # the collate function does the processing
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collate_fn,
    processing_class=processor,  # the full processor, so TRL treats the model as a VLM
    callbacks=[TrainingHealthCallback(OUTPUT_DIR)],
)
trainer.train()
final_loss_metrics = trainer.evaluate()  # scheduled steps may miss the final optimizer step
trainer.save_model(OUTPUT_DIR)
print(f"💾 LoRA adapter saved to {OUTPUT_DIR}")

## Inspect training health

This cell saves `training_log.csv` and `training_health.png` in the output folder. Check the validation-loss trend, gradient norm, and GPU memory before comparing epoch checkpoints by detection F1. A falling final validation loss can justify *testing* another epoch; a rising loss while training loss falls suggests overfitting. Neither trend alone is a reason to use a checkpoint without checking box F1.


In [ ]:
import json
import math
import pandas as pd
import matplotlib.pyplot as plt

log_path = Path(OUTPUT_DIR) / "training_log.jsonl"
assert log_path.exists(), f"Missing training log: {log_path}"
history = pd.DataFrame(json.loads(line) for line in log_path.read_text().splitlines() if line.strip())
nonfinite_counts = {}
for metric in ("loss", "eval_loss", "grad_norm"):
    if metric in history:
        nonfinite_counts[metric] = sum(
            not math.isfinite(float(value)) for value in history[metric] if not pd.isna(value)
        )
        history[metric] = pd.to_numeric(history[metric], errors="coerce")
history.to_csv(Path(OUTPUT_DIR) / "training_log.csv", index=False)
train_log = history.loc[history["loss"].map(math.isfinite)] if "loss" in history else pd.DataFrame()
eval_log = history.loc[history["eval_loss"].map(math.isfinite)] if "eval_loss" in history else pd.DataFrame()

print("Validation-loss checkpoints (fixed 200-image subset):")
display(eval_log.reindex(columns=["step", "epoch", "eval_loss", "eval_runtime", "gpu_peak_allocated_gb"]).to_string(index=False))
print("Most recent training logs:")
display(train_log.reindex(columns=["step", "epoch", "loss", "grad_norm", "learning_rate", "gpu_allocated_gb"]).tail(8).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if not train_log.empty:
    axes[0].plot(train_log["step"], train_log["loss"], label="training loss")
if not eval_log.empty:
    axes[0].plot(eval_log["step"], eval_log["eval_loss"], "o-", label="validation loss")
axes[0].set(xlabel="optimizer step", ylabel="answer-token loss", title="Loss")
axes[0].legend()
if "grad_norm" in train_log:
    finite_grad = train_log.loc[train_log["grad_norm"].map(lambda value: pd.notna(value) and math.isfinite(value))]
    axes[1].plot(finite_grad["step"], finite_grad["grad_norm"])
axes[1].set(xlabel="optimizer step", ylabel="gradient norm", title="Gradient health")
fig.tight_layout()
plot_path = Path(OUTPUT_DIR) / "training_health.png"
fig.savefig(plot_path, dpi=150)
plt.show()

for metric, count in nonfinite_counts.items():
    if count:
        print(f"WARNING: {metric} has {count} non-finite log entries.")
if len(eval_log) >= 2:
    change = float(eval_log.iloc[-1]["eval_loss"] - eval_log.iloc[-2]["eval_loss"])
    print(f"Final validation-loss change: {change:+.4f} (last two checks). Check detection F1 before choosing more epochs.")
print(f"Saved {log_path}, {Path(OUTPUT_DIR) / 'training_log.csv'}, and {plot_path}")


## Check on the Validation Split

Score all 720 validation images with the trained adapter. Compare the saved epoch checkpoints with the same `be.run_local` settings. Each results file name includes a hash of the run configuration, so results from different settings never mix.


In [ ]:
import pandas as pd

model.eval()
run_name = f"lfm3-vl-2b-ft-{os.path.basename(OUTPUT_DIR.rstrip('/'))}-val"
val_eval = be.load_items(root, split="val", n=N_VAL_EVAL, exclude_cross_class_duplicates=True)
config = {
    "base_model": BASE_MODEL_INFO,
    "adapter": be.model_identity(OUTPUT_DIR),
    "split": "val",
    "dataset_url": be.DATASET_URL,
    "exclude_cross_class_duplicates": True,
    "max_image_tokens": MAX_IMAGE_TOKENS,
}
path = be.run_local(model, processor, val_eval, run_name, BOX_FORMAT, results_dir="results_val", run_config=config)
records = be.read_jsonl(path)
display(be.summarize(records).round(3))
display(be.recall_by_size(records).round(3))
print("F1 by the image's annotated class (diagnostic slice):")
display(be.per_class(records).round(2))


Ground truth in green, predictions in red:

In [ ]:
lookup = {item.item_id: item for item in val_eval}
sample = random.Random(0).sample(records, 4)
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, record in zip(axes, sample):
    ax.imshow(be.draw(lookup[record["item_id"]], record))
    ax.set_title(f"{record['label']}\nTP {record['tp']}  FP {record['fp']}  FN {record['fn']}", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Back up the final and epoch adapters

Colab storage disappears when the runtime ends. This cell packages the LoRA adapter files and the training logs from the final folder and each epoch checkpoint, without the large optimizer states. Keep the ZIP even if you push the final adapter to the Hub: it holds both epoch checkpoints for comparison.

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_STORED

output = Path(OUTPUT_DIR)
adapter_dirs = [output] + sorted(output.glob("checkpoint-*"))
archive = Path(f"{output.name}-adapters.zip")
with ZipFile(archive, "w", compression=ZIP_STORED) as bundle:
    for folder in adapter_dirs:
        files = sorted(folder.glob("adapter_model*.safetensors")) + sorted(folder.glob("adapter_model*.bin")) + sorted(folder.glob("adapter_config.json"))
        assert any(file.name.startswith("adapter_model") for file in files) and any(file.name == "adapter_config.json" for file in files), f"Missing adapter files in {folder}"
        for file in files:
            bundle.write(file, arcname=file.relative_to(output.parent))
    for name in ("training_log.jsonl", "training_log.csv", "training_health.png"):
        file = output / name
        if file.exists():
            bundle.write(file, arcname=file.relative_to(output.parent))
print(f"Saved {len(adapter_dirs)} adapter folders in {archive} ({archive.stat().st_size / 1024**2:.1f} MiB). Keep this ZIP.")
try:
    from google.colab import files as colab_files
except ImportError:
    print(f"Local run: copy {archive} somewhere durable.")
else:
    colab_files.download(str(archive))

## Push the adapter

Pushing the final adapter to the Hub (a few tens of MB) makes it available outside Colab, and [`lfm3_vl_grpo.ipynb`](lfm3_vl_grpo.ipynb) loads it from there. Set `HUB_REPO` in the configuration first.

In [ ]:
if HUB_REPO:
    model.push_to_hub(HUB_REPO, private=True)
    print(f"Pushed the adapter to https://huggingface.co/{HUB_REPO}")
else:
    print(f"HUB_REPO not set; the adapter is only in {OUTPUT_DIR}. Download it before the Colab runtime ends.")

## Next steps

- **Pick a checkpoint:** compare the final adapter and the saved `checkpoint-*` adapters on the same 720 validation images, using detection F1 first and then precision, recall, type accuracy and parse rate. In our run, epoch 2 scored best (detection F1 0.089 against 0.071 after epoch 1).
- **Continue with reinforcement learning:** [`lfm3_vl_grpo.ipynb`](lfm3_vl_grpo.ipynb) starts from this adapter and trains on a reward computed from the predicted boxes.
- **Test once, at the end:** score the held-out test split with [`evaluate.ipynb`](evaluate.ipynb) only after you've picked your final model.